<a href="https://colab.research.google.com/github/davis-mironga/wastewater-treatment-analytics/blob/main/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Data Audit: UCI Water Treatment Plant

**Purpose:** Find out exactly what is in the raw data (size, structure, missing values, quirks) before any analysis.

**Why first:** Every later result depends on the data being what we think it is. Raw files are read only and never modified.

## 1. Setup: get the project into Colab
Colab starts with an empty workspace. We clone the public repo so the raw data is available, then move into the project folder so paths match the repo layout.

In [ ]:
import os
if not os.path.exists("data/raw"):
    !git clone https://github.com/davis-mironga/wastewater-treatment-analytics.git
    %cd wastewater-treatment-analytics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("data/raw")
print(sorted(p.name for p in RAW.iterdir()))

Cloning into 'wastewater-treatment-analytics'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 35 (delta 6), reused 27 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 46.51 KiB | 6.64 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/wastewater-treatment-analytics
['.gitkeep', 'CHECKSUMS.txt', 'water-treatment.data', 'water-treatment.names']


### What we see
*(Which files exist? Do they match what we downloaded?)*

## 2. Read the dictionary's variable descriptions
We read what the original documentation says each variable means, and its units, before labelling columns.

In [ ]:
with open(RAW / "water-treatment.names") as f:
    lines = f.readlines()
print("".join(lines[59:140]))

7. Attribute Information:

 All atrributes are numeric and continuous

N.  Attrib.    
 1  Q-E        (input flow to plant)  
 2  ZN-E       (input Zinc to plant)
 3  PH-E       (input pH to plant) 
 4  DBO-E      (input Biological demand of oxygen to plant) 
 5  DQO-E      (input chemical demand of oxygen to plant)
 6  SS-E       (input suspended solids to plant)  
 7  SSV-E      (input volatile supended solids to plant)
 8  SED-E      (input sediments to plant) 
 9  COND-E     (input conductivity to plant) 
10  PH-P       (input pH to primary settler)
11  DBO-P      (input Biological demand of oxygen to primary settler)
12  SS-P       (input suspended solids to primary settler)
13  SSV-P      (input volatile supended solids to primary settler)
14  SED-P      (input sediments to primary settler) 
15  COND-P     (input conductivity to primary settler)
16  PH-D       (input pH to secondary settler) 
17  DBO-D      (input Biological demand of oxygen to secondary settler)
18  DQO-D      (

### What we see
The loaded file matches the documentation's missing-value table exactly
for all 38 variables. The highest missing share is RD-DBO-P at 11.8%.
No variable is too sparse to use.

## 3. Load the data
From the raw-file checks: no header row, `?` marks missing values, first column is a date, 69 blank lines (pandas skips these). Column names come from the dictionary's own list.

In [ ]:
cols = ["date",
 "Q-E","ZN-E","PH-E","DBO-E","DQO-E","SS-E","SSV-E","SED-E","COND-E",
 "PH-P","DBO-P","SS-P","SSV-P","SED-P","COND-P",
 "PH-D","DBO-D","DQO-D","SS-D","SSV-D","SED-D","COND-D",
 "PH-S","DBO-S","DQO-S","SS-S","SSV-S","SED-S","COND-S",
 "RD-DBO-P","RD-SS-P","RD-SED-P","RD-DBO-S","RD-DQO-S",
 "RD-DBO-G","RD-DQO-G","RD-SS-G","RD-SED-G"]

df = pd.read_csv(RAW / "water-treatment.data",
                 header=None, names=cols, na_values="?")
print(df.shape)
df.head()

(527, 39)


,date,Q-E,ZN-E,PH-E,DBO-E,DQO-E,SS-E,SSV-E,SED-E,COND-E,...,COND-S,RD-DBO-P,RD-SS-P,RD-SED-P,RD-DBO-S,RD-DQO-S,RD-DBO-G,RD-DQO-G,RD-SS-G,RD-SED-G
0,D-1/3/90,44101.0,1.5,7.8,NaN,407.0,166.0,66.3,4.5,2110,...,2000.0,NaN,58.8,95.5,NaN,70.0,NaN,79.4,87.3,99.6
1,D-2/3/90,39024.0,3.0,7.7,NaN,443.0,214.0,69.2,6.5,2660,...,2590.0,NaN,60.7,94.8,NaN,80.8,NaN,79.5,92.1,100.0
2,D-4/3/90,32229.0,5.0,7.6,NaN,528.0,186.0,69.9,3.4,1666,...,1888.0,NaN,58.2,95.6,NaN,52.9,NaN,75.8,88.7,98.5
3,D-5/3/90,35023.0,3.5,7.9,205.0,588.0,192.0,65.6,4.5,2430,...,1840.0,33.1,64.2,95.3,87.3,72.3,90.2,82.3,89.6,100.0
4,D-6/3/90,36924.0,1.5,8.0,242.0,496.0,176.0,64.8,4.0,2110,...,2120.0,NaN,62.7,95.6,NaN,71.0,92.1,78.2,87.5,99.5


### What we see
*(Expected shape: 527 rows × 39 columns.)*

## 4. Structure checks
Confirm column types and parse the date. Dates are stored as text like
`D-1/3/90` (day/month/year). We parse into a new column and leave the
original untouched, then check the date range, duplicates and gaps.

In [ ]:
df.info()

df["date_parsed"] = pd.to_datetime(
    df["date"].str.replace("D-", "", regex=False), format="%d/%m/%y")

print(df["date_parsed"].min(), "to", df["date_parsed"].max())
print("duplicate dates:", df["date_parsed"].duplicated().sum())
gaps = df["date_parsed"].sort_values().diff().dt.days
print(gaps.value_counts().sort_index().head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 527 entries, 0 to 526
Data columns (total 39 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      527 non-null    object 
 1   Q-E       509 non-null    float64
 2   ZN-E      524 non-null    float64
 3   PH-E      527 non-null    float64
 4   DBO-E     504 non-null    float64
 5   DQO-E     521 non-null    float64
 6   SS-E      526 non-null    float64
 7   SSV-E     516 non-null    float64
 8   SED-E     502 non-null    float64
 9   COND-E    527 non-null    int64  
 10  PH-P      527 non-null    float64
 11  DBO-P     487 non-null    float64
 12  SS-P      527 non-null    int64  
 13  SSV-P     516 non-null    float64
 14  SED-P     503 non-null    float64
 15  COND-P    527 non-null    int64  
 16  PH-D      527 non-null    float64
 17  DBO-D     499 non-null    float64
 18  DQO-D     518 non-null    float64
 19  SS-D      525 non-null    float64
 20  SSV-D     514 non-null    float6

### What we see
The file loaded as 527 rows × 39 columns (date + 38 attributes), matching
both the documentation and the raw-file checks. Missing values appear as
NaN. In the first rows, the RD- (removal) columns are missing whenever the
BOD measurements they depend on are missing. To be tested below.

## 4. Structure checks
Confirm column types and parse the date. Dates are stored as text like
`D-1/3/90` (day/month/year). We parse into a new column and leave the
original untouched, then check the date range, duplicates and gaps.

In [ ]:
df.info()

df["date_parsed"] = pd.to_datetime(
    df["date"].str.replace("D-", "", regex=False), format="%d/%m/%y")

print(df["date_parsed"].min(), "to", df["date_parsed"].max())
print("duplicate dates:", df["date_parsed"].duplicated().sum())
gaps = df["date_parsed"].sort_values().diff().dt.days
print(gaps.value_counts().sort_index().head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 527 entries, 0 to 526
Data columns (total 40 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         527 non-null    object        
 1   Q-E          509 non-null    float64       
 2   ZN-E         524 non-null    float64       
 3   PH-E         527 non-null    float64       
 4   DBO-E        504 non-null    float64       
 5   DQO-E        521 non-null    float64       
 6   SS-E         526 non-null    float64       
 7   SSV-E        516 non-null    float64       
 8   SED-E        502 non-null    float64       
 9   COND-E       527 non-null    int64         
 10  PH-P         527 non-null    float64       
 11  DBO-P        487 non-null    float64       
 12  SS-P         527 non-null    int64         
 13  SSV-P        516 non-null    float64       
 14  SED-P        503 non-null    float64       
 15  COND-P       527 non-null    int64         
 16  PH-D    

## 5. Missing values: does the file match the documentation?
The `.names` file lists how many values are missing per variable. We count
them in the loaded data and compare. Agreement shows the file loaded
correctly. Disagreement is a finding to record.

In [ ]:
documented = {
 "Q-E":18,"ZN-E":3,"PH-E":0,"DBO-E":23,"DQO-E":6,"SS-E":1,"SSV-E":11,
 "SED-E":25,"COND-E":0,"PH-P":0,"DBO-P":40,"SS-P":0,"SSV-P":11,"SED-P":24,
 "COND-P":0,"PH-D":0,"DBO-D":28,"DQO-D":9,"SS-D":2,"SSV-D":13,"SED-D":25,
 "COND-D":0,"PH-S":1,"DBO-S":23,"DQO-S":18,"SS-S":5,"SSV-S":17,"SED-S":28,
 "COND-S":1,"RD-DBO-P":62,"RD-SS-P":4,"RD-SED-P":27,"RD-DBO-S":40,
 "RD-DQO-S":26,"RD-DBO-G":36,"RD-DQO-G":25,"RD-SS-G":8,"RD-SED-G":31}

actual = df[cols[1:]].isna().sum()
comp = pd.DataFrame({"documented": pd.Series(documented), "actual": actual})
comp["diff"] = comp["actual"] - comp["documented"]
comp["pct_missing"] = (comp["actual"] / len(df) * 100).round(1)

print("Columns where file and documentation disagree:")
print(comp[comp["diff"] != 0])
print()
print(comp.sort_values("pct_missing", ascending=False).head(10))

Columns where file and documentation disagree:
Empty DataFrame
Columns: [documented, actual, diff, pct_missing]
Index: []

          documented  actual  diff  pct_missing
RD-DBO-P          62      62     0         11.8
DBO-P             40      40     0          7.6
RD-DBO-S          40      40     0          7.6
RD-DBO-G          36      36     0          6.8
RD-SED-G          31      31     0          5.9
DBO-D             28      28     0          5.3
SED-S             28      28     0          5.3
RD-SED-P          27      27     0          5.1
RD-DQO-S          26      26     0          4.9
RD-DQO-G          25      25     0          4.7


### What we see
The loaded file matches the documentation's missing-value table exactly
for all 38 variables, so the `?` marker was read correctly and no column
shifted. Missing values are modest: the highest is RD-DBO-P at 11.8%, and
the BOD variables are the most affected. No variable is too sparse to use.

## Audit summary
- The file loaded as 527 rows × 39 columns (date + 38 variables), matching the documentation.
- The `?` marker was read correctly. Missing counts match the documentation for all 38 variables, and none exceeds 12%.
- Dates parse correctly with no duplicates. They span 1 Jan 1990 to 30 Oct 1991, but only 527 days have records.
- The dictionary gives no units, and the RD- descriptions are ambiguous. Both need handling in the analysis plan.
- **Conclusion:** the data is usable and matches its documentation. No rows were removed or changed.